# Image encoder — Sérsic linear probes

Frozen `jwst_dino` teacher on COSMOS-Web F150W cutouts, ridge probes for Sérsic
index, effective radius and axis ratio. Writes `sersic.png`.

In [ ]:
import sys, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.simplefilter("ignore")

from plotstyle import (IMAGE_DIR, IMAGE_ROOT, COSMOS_PHOTOM, IMG_CKPT,
                       C_IMAGE, C_KNN, C_THIRD, use_style, style_axes, fs, save)
sys.path.insert(0, str(IMAGE_DIR))
from model.jwst_dino import load_teacher_backbone
from data.augmentations import AsinhStretch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sample definition is purely observational: resolved, detected, and with a fit
# that did not run into a bound. Label uncertainty is not used to select sources
# (it tracks S/N with correlation 0.9 and additionally biases against flat
# sources, because a fixed relative error is harder to reach at small axis ratio).
PSF_FWHM_PX = 0.049 / 0.030   # F150W FWHM at 30 mas/pix
MIN_REFF_PX = 1 * PSF_FWHM_PX  # resolved: Re > 1 PSF FWHM
MIN_SNR = 30.0                 # F150W detection significance
MAX_SAMPLES = 30000
TEST_FRAC, SEED = 0.5, 42
ALPHAS = np.logspace(-8, 6, 43)


def alpha_edge(a, grid=ALPHAS):
    """Flag a penalty that sits on an end of the search grid."""
    return ("FLOOR" if a <= grid[0] * 1.001 else
            "CEIL" if a >= grid[-1] * 0.999 else "ok")

# (photometry column, axis label, fit in log10?)
TARGETS = [("sersic",         r"$\log_{10} n$",         True),
           ("radius_sersic",  r"$\log_{10}\,R_e$ [px]", True),
           ("axratio_sersic", r"axis ratio $q$",         False)]
# A value pinned at a fitter bound is a failed fit, not a measurement.
RAIL_BOUNDS = {"sersic": (0.31, 8.30), "axratio_sersic": (0.05, 0.95)}

use_style(1.75)
net = load_teacher_backbone(str(IMG_CKPT), DEVICE)
CROP = net.crop_size
print(f"device {DEVICE}  crop {CROP}px")

## Sample

COSMOS-Web cutouts with the catalogue's single-Sérsic fits, cut on
resolvedness, F150W detection significance and fitter rails.


In [ ]:
DEG_TO_PIX = 3600 * 1000 / 30.0   # radius_sersic [deg] -> px at 30 mas/pix


class CosmosSersicDataset(Dataset):
    """COSMOS F150W cutouts with single-Sérsic labels, stacked (N, 3).

    The image index `id` is the photometry row index, so labels are read by
    direct indexing. A source is kept when it is resolved, detected above
    min_snr, and none of its parameters sits on a fitter bound.
    """

    def __init__(self, root, photom, targets, rail_bounds, filt="f150w", crop_size=72,
                 min_reff_px=3.27, min_snr=30.0, max_samples=-1, Q=20.0, scale=1.0,
                 seed=42):
        self.root = root
        self.center_crop = transforms.CenterCrop(crop_size)
        self.stretch = AsinhStretch(scale=scale, Q=Q, return_channel_pos=0)
        self.shards = {}

        index = Table.read(root / f"image_index_cosmos_{filt}.fits")
        ids = np.asarray(index["id"], np.int64)
        rel_path = np.asarray(index["rel_path"]).astype(str)
        local_idx = np.asarray(index["local_idx"], np.int64)

        ph = fits.open(photom, memmap=True)[1].data
        get = lambda c: np.asarray(ph[c], np.float64)[ids]
        val = {"sersic": get("sersic"),
               "radius_sersic": get("radius_sersic") * DEG_TO_PIX,
               "axratio_sersic": get("axratio_sersic")}
        err = {"sersic": get("sersic_err"),
               "radius_sersic": get("radius_sersic_err") * DEG_TO_PIX,
               "axratio_sersic": get("axratio_sersic_err")}
        snr = np.asarray(ph["snr_" + filt], np.float64)[ids]

        resolved = (np.isfinite(val["radius_sersic"]) & (val["radius_sersic"] >= min_reff_px)
                    & np.isfinite(val["sersic"]) & np.isfinite(val["axratio_sersic"])
                    & (val["axratio_sersic"] > 0))
        railed = np.zeros(len(ids), bool)
        for c, (lo, hi) in rail_bounds.items():
            railed |= (val[c] <= lo) | (val[c] >= hi)
        detected = np.isfinite(snr) & (snr > min_snr)

        clean = resolved & ~railed & detected
        keep = np.where(clean)[0]
        if 0 < max_samples < len(keep):
            keep = np.random.default_rng(seed).choice(keep, max_samples, replace=False)

        labels = np.stack([np.log10(val[c]) if log else val[c]
                           for c, _, log in targets], 1).astype(np.float32)
        self._samples = [(rel_path[i], local_idx[i]) for i in keep]
        self._labels = labels[keep]
        print(f"{len(ids)} cutouts -> {resolved.sum()} resolved (Re > {min_reff_px:.2f} px) "
              f"-> {clean.sum()} with S/N > {min_snr:g} and no railed parameter "
              f"-> {len(keep)} used")

    def _shard(self, rp):
        if rp not in self.shards:
            self.shards[rp] = np.load(self.root / rp, mmap_mode="r")
        return self.shards[rp]

    def __len__(self):
        return len(self._samples)

    def __getitem__(self, i):
        rp, li = self._samples[i]
        img = np.nan_to_num(self._shard(rp)[li].astype(np.float32))[None]
        img = self.center_crop(torch.from_numpy(img)).numpy()
        return torch.from_numpy(self.stretch(img)), self._labels[i]


ds = CosmosSersicDataset(IMAGE_ROOT, COSMOS_PHOTOM, TARGETS, RAIL_BOUNDS,
                         crop_size=CROP, min_reff_px=MIN_REFF_PX, min_snr=MIN_SNR,
                         max_samples=MAX_SAMPLES, seed=SEED)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

## Embeddings and probe

Readout is `concat(CLS, mean patch token)`; one closed-form ridge head per target.

In [ ]:
@torch.no_grad()
def extract(net, loader, device):
    embs, labels = [], []
    for imgs, ys in tqdm(loader, desc="embeddings"):
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=device.type == "cuda"):
            out = net(imgs.to(device))
        embs.append(torch.cat([out["cls"], out["patch"].mean(1)], 1).float().cpu().numpy())
        labels.append(np.asarray(ys))
    return np.concatenate(embs), np.concatenate(labels)


X, Y = extract(net, loader, DEVICE)
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=TEST_FRAC, random_state=SEED)
sc = StandardScaler().fit(Xtr)
Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)

preds = np.zeros_like(Yte)
print(f"{len(Xtr)} train / {len(Xte)} test, embedding {X.shape[1]}d\n")
print(f"{'target':16s} {'R2':>7s} {'sigma_NMAD':>11s} {'alpha':>10s}")
for j, (col, _, _) in enumerate(TARGETS):
    r = RidgeCV(alphas=ALPHAS).fit(Ztr, Ytr[:, j])
    preds[:, j] = r.predict(Zte)
    d = preds[:, j] - Yte[:, j]
    print(f"{col:16s} {r2_score(Yte[:, j], preds[:, j]):7.3f} "
          f"{1.4826 * np.median(np.abs(d - np.median(d))):11.4f} "
          f"{r.alpha_:10.2e} {alpha_edge(r.alpha_)}")

## Zero-shot k-NN

Non-parametric readout on the same embeddings and the same split, quoted in the
text alongside the linear probe.

In [ ]:
# k=5 neighbour average on the standardized embeddings; distances on the GPU in
# chunks, since a 15k x 15k dense distance matrix is the whole cost here.
KNN_K, CHUNK = 5, 2048


@torch.no_grad()
def knn_predict(Ztr, Zte, Ytr, k=KNN_K):
    A = torch.as_tensor(Ztr, dtype=torch.float32, device=DEVICE)
    B = torch.as_tensor(Zte, dtype=torch.float32, device=DEVICE)
    L = torch.as_tensor(Ytr, dtype=torch.float32, device=DEVICE)
    a2 = (A * A).sum(1)                      # ||a||^2; the ||b||^2 term is constant per row
    out = []
    for s in range(0, B.shape[0], CHUNK):
        b = B[s:s + CHUNK]
        score = -(a2[None, :] - 2.0 * (b @ A.T))
        out.append(L[score.topk(k, dim=1).indices].mean(1).cpu().numpy())
    return np.concatenate(out)


knn_pred = knn_predict(Ztr, Zte, Ytr)
print(f"{'target':16s} {'R2 knn':>8s} {'R2 ridge':>9s}")
for j, (col, _, _) in enumerate(TARGETS):
    print(f"{col:16s} {r2_score(Yte[:, j], knn_pred[:, j]):8.3f} "
          f"{r2_score(Yte[:, j], preds[:, j]):9.3f}")
print(f"\nmean R2   zero-shot {np.mean([r2_score(Yte[:, j], knn_pred[:, j]) for j in range(3)]):.3f}"
      f"   linear {np.mean([r2_score(Yte[:, j], preds[:, j]) for j in range(3)]):.3f}")

## Figure

In [ ]:
def sigma_nmad(t, p):
    d = p - t
    return 1.4826 * np.median(np.abs(d - np.median(d)))


PANEL_C = [C_IMAGE, C_KNN, C_THIRD]
PAD = 0.05   # axis margin as a fraction of the plotted range

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4.8))
for j, (ax, (col, label, _)) in enumerate(zip(axes, TARGETS)):
    t, p, c = Yte[:, j], preds[:, j], PANEL_C[j % len(PANEL_C)]
    lo, hi = np.percentile(np.concatenate([t, p]), [0.5, 99.5])
    m = PAD * (hi - lo)
    lo, hi = lo - m, hi + m

    ax.scatter(t, p, s=6, alpha=0.2, edgecolors="none", color=c)
    ax.plot([lo, hi], [lo, hi], "--", lw=2, color="black", alpha=0.4, label="1:1")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_box_aspect(1)
    ax.set_xlabel(rf"true  {label}", fontsize=fs(10))
    ax.set_ylabel(rf"predicted  {label}", fontsize=fs(10))
    ax.text(0.97, 0.03, rf"$R^2$={r2_score(t, p):.3f}" + "\n"
            + rf"$\sigma_{{\rm NMAD}}$={sigma_nmad(t, p):.4f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=fs(9), color=c,
            fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.7, pad=3.0, edgecolor="none"))
    if j == 0:
        ax.legend(fontsize=fs(12), loc="upper left", frameon=False)
    style_axes(ax)

fig.patch.set_alpha(0.0)
fig.tight_layout()
save(fig, "sersic")
plt.show()